In [ ]:
# Paths come from the memtune package (see docs/INSTALL.md): no machine-specific locations.
#   data   -> data/lbfgs_runs.csv   (override: MEMTUNE_DATA_DIR)
#   models -> models/              (override: MEMTUNE_MODELS_DIR)
#   output -> results/             (override: MEMTUNE_RESULTS_DIR)
from memtune import paths
(paths.results_dir() / "retrained_models").mkdir(parents=True, exist_ok=True)  # notebooks never overwrite models/


In [ ]:
import pandas as pd
import numpy as np
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import RandomForestRegressor
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 20)
from sklearn.model_selection import RandomizedSearchCV
import numpy as np
from scipy.stats import randint, uniform
import joblib
import csv

### Data Preprocessing

In [ ]:
df = pd.read_csv(str(paths.dataset_path()))
first = ["name", "nvar"]
df = df[first + [c for c in df.columns if c not in first]]
df["nvar"].value_counts
df = df.sort_values(["name", "nvar", "mem"]).reset_index()
df

In [ ]:
feature_cols_reg = [
                    # Core Features
                    "nvar", 
                    "mem",
                    "tree_length", 
                    "tree_depth", 
                    "time_obj",  # initial eval
                    "time_grad",
                    "extract_duration_problem",
                    
                    # Expression Tree Features
                    "highest_degree (ExprTree)",
                    "count_plus_minus",
                    "count_trigonometric_function",
                    "count_exponential_function",
                    "count_op_function",

                    # allocation
                    "alloc_obj",
                    "alloc_grad",
                    "total_alloc",
                  ]

target_cols_reg = [ "neval_obj",
                    "neval_grad",
                    "timed_bytes"] #(neval_obj, neval_grad, timed_bytes)

target_col_model = ["stats_elapsed_time"]

group_key = ["name", "nvar"]                                                                                                                   # [...]     
all_cols = list(set(feature_cols_reg + target_cols_reg + target_col_model + group_key))
solver_metrics = ['neval_grad', 'neval_obj', 'timed_bytes']
df[solver_metrics] = df[solver_metrics].fillna(0)
df = df[all_cols]
df.isna().sum().sort_values(ascending=False)

In [ ]:
df = df.copy()
df["mem_count"] = df.groupby(["name", "nvar"]).transform('size')
df = df[df["mem_count"] == 100].copy().reset_index(drop=True)
df

In [ ]:
# unique instances: one row per (problem, nvar)
instances = df[group_key].drop_duplicates()

# shuffle instances
instances = instances.sample(frac=1, random_state=66).reset_index(drop=True)

n = len(instances)
n_train = int(0.8 * n)

train_inst = instances.iloc[:n_train]
test_inst  = instances.iloc[n_train :]

# assign rows to splits by (problem, nvar)
train_df = df.merge(train_inst, on=["nvar", "name"], how="inner").reset_index(drop=True)
test_df  = df.merge(test_inst,  on=["nvar", "name"], how="inner").reset_index(drop=True)
all_df = pd.concat([train_df, test_df], axis=0)
all_df = all_df.reset_index(drop=True)
all_df

In [ ]:
 # confirm and use the three predicted feature to predict time                                                                                                              #  
X_train = train_df[feature_cols_reg].to_numpy(dtype=float)
X_test  = test_df[feature_cols_reg].to_numpy(dtype=float)

y_train = np.log1p(train_df[target_cols_reg].to_numpy(dtype=float))
y_test  = np.log1p(test_df[target_cols_reg].to_numpy(dtype=float))

X_all = np.concatenate([X_train, X_test], axis=0)


In [ ]:
X_train_time = train_df[target_cols_reg]
X_test_time = test_df[target_cols_reg]
y_train_time = np.log1p(train_df[target_col_model].to_numpy(dtype=float)).ravel()
y_test_time = np.log1p(test_df[target_col_model].to_numpy(dtype=float)).ravel()

### Global Performance Ratio Function

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import gmean

def evaluate_model(model, label="Model"):
    """
    Evaluates a stage-2 model using global X_test_time and global test_df.
    Includes arithmetic mean, geometric mean, and maximum performance ratio metrics.
    """
    predictions = model.predict(X_test_time)
    working_df = test_df.copy() 
    
    working_df["predicted_time"] = predictions
    group_cols = ["name", "nvar"]

    # 1. Model Selection
    model_choices = (
        working_df.sort_values(by=group_cols + ["predicted_time", "mem"], ascending=[True, True, True, False])
        .groupby(group_cols).first().reset_index()
        [group_cols + ["mem", "predicted_time", "stats_elapsed_time"]]
        .rename(columns={"mem": "predicted_mem", "stats_elapsed_time": "actual_time"})
    )

    # 2. Oracle Selection
    oracle_choices = (
        working_df.sort_values(by=group_cols + ["stats_elapsed_time", "mem"], ascending=[True, True, True, False])
        .groupby(group_cols).first().reset_index()
        [group_cols + ["mem", "stats_elapsed_time"]]
        .rename(columns={"mem": "best_mem", "stats_elapsed_time": "best_time"})
    )

    # 3. Merge Matrix & Apply Hardened 0.10s Absolute Tolerance Filter
    final_df = pd.merge(model_choices, oracle_choices, on=group_cols).copy()
    final_df['actual_time'] = pd.to_numeric(final_df['actual_time'], errors='coerce')
    final_df['best_time'] = pd.to_numeric(final_df['best_time'], errors='coerce')

    final_df['raw_ratio'] = final_df['actual_time'] / final_df['best_time']
    time_diff = np.abs(final_df['actual_time'] - final_df['best_time'])
    final_df['performance_ratio'] = np.where(time_diff <= 0.10, 1.0, final_df['raw_ratio'])

    # Reorder display columns
    column_order = group_cols + ["predicted_mem", "predicted_time", "actual_time", "best_mem", "best_time", "performance_ratio", "raw_ratio"]
    final_df = final_df[column_order]

    # Output Summary Blocks
    print("\n" + "="*145)
    print(f"PER-PROBLEM PERFORMANCE PROFILE SUMMARY ({label.upper()})")
    print("="*145)
    print(final_df.to_string(index=False, formatters={
        'predicted_time': '{:.4f}'.format, 'actual_time': '{:.4f}'.format,
        'best_time': '{:.4f}'.format, 'performance_ratio': '{:.4f}'.format, 'raw_ratio': '{:.4f}'.format
    }))
    print("="*145)

    total_actual = final_df['actual_time'].sum()
    total_best = final_df['best_time'].sum()
    
    # Calculate Core Evaluation Statistics
    global_mean_ratio = final_df['performance_ratio'].mean()
    geom_mean_ratio = gmean(final_df['performance_ratio'])  # Clean Scipy implementation
    max_performance_ratio = final_df['performance_ratio'].max()
    
    print("\n" + "="*50)
    print(f"AGGREGATE METRICS ({label.upper()})")
    print("="*50)
    print(final_df['performance_ratio'].quantile([0.50, 0.75, 0.90]).to_string(float_format='{:.4f}'.format))
    print("-"*50)
    print(f"Global Mean Perf. Ratio     : {global_mean_ratio:.4f}")
    print(f"Geometric Mean Perf. Ratio  : {geom_mean_ratio:.4f}")
    print(f"Maximum Performance Ratio   : {max_performance_ratio:.4f}")
    print("-"*50)
    print(f"Total Actual Compute Time   : {total_actual:.4f}s")
    print(f"Total Oracle Optimal Time   : {total_best:.4f}s")
    print("="*50)
    
    return final_df

### Linear Regression Stage 1

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict
import joblib  # Standard library for saving scikit-learn models

lr = LinearRegression()

# 1. Cross-Validation Step (on Training Data)
cv_pred = cross_val_predict(lr, X_train, y_train, cv=5)
cv_mse_per_target = mean_squared_error(y_train, cv_pred, multioutput='raw_values')
cv_r2_per_target = r2_score(y_train, cv_pred, multioutput='raw_values')

print("--- 5-Fold Cross-Validation Metrics (Train Set) ---")
for i, col in enumerate(target_cols_reg):
    print(f"{col}: CV MSE = {cv_mse_per_target[i]:.4f} | CV R² = {cv_r2_per_target[i]:.4f}")

# 2. Final Fit and Test Evaluation
lr.fit(X_train, y_train)

# --- SAVE MODEL WEIGHTS HERE ---
joblib.dump(lr, str(paths.results_dir() / "retrained_models" / "lr_stage1_model.joblib"))
print("\n[INFO] Stage 1 Linear Regression model saved successfully to disk.")

# Generate Stage 1 predictions on the test set
pred = lr.predict(X_test)

# PIPELINE LINK: If X_test here matches your Stage 2 test rows, 
# this 'pred' array is exactly your 'pred_lr_s1' for Stage 2.
pred_lr_s1 = pred 

test_mse_per_target = mean_squared_error(y_test, pred, multioutput='raw_values')
test_r2_per_target = r2_score(y_test, pred, multioutput='raw_values')

print("\n--- Final Test Set Metrics ---")
for i, col in enumerate(target_cols_reg):
    print(f"{col}: Test MSE = {test_mse_per_target[i]:.4f} | Test R² = {test_r2_per_target[i]:.4f}")

# 3. Quantiled Relative Error Distribution
relative_error = np.abs(pred - y_test) / (np.abs(y_test) + 1e-8)

quantiles = [0.25, 0.50, 0.75]
print("\n--- Quantiled Relative Error Q(q) (Test Set) ---")
for i, col in enumerate(target_cols_reg):
    print(f"\n{col}:")
    print(f"Quantile (q) | Relative Error Value Q(q)")
    error_data = relative_error[:, i] if relative_error.ndim > 1 else relative_error
    q_values = np.quantile(error_data, quantiles)
    for q, val in zip(quantiles, q_values):
        print(f"  Q({q:.2f}):  {val:.4f}")

### Linear Regression Stage 2 using predicted value from Stage 1 Linear Regression

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_predict

# 1. Define targets cleanly in log-space first
y_train_time = np.log1p(train_df[target_col_model].to_numpy(dtype=float)).ravel()
y_test_time = np.log1p(test_df[target_col_model].to_numpy(dtype=float)).ravel()

# 2. INTRODUCE ERROR: Generate out-of-fold predictions for the training set (in log-space)
X_train_time = cross_val_predict(lr, X_train, y_train_time, cv=5, n_jobs=-1)

# 3. Fit the base model on the full training set to prepare it for test predictions
lr.fit(X_train, y_train_time)

# 4. Generate standard predictions for the test set (in log-space)
X_test_time = lr.predict(X_test)

In [ ]:
# 1. Load Stage 1 model and generate raw log-space feature predictions
lr_stage1_model = joblib.load(str(paths.results_dir() / "retrained_models" / "lr_stage1_model.joblib"))
pred_train_log = lr_stage1_model.predict(X_train)
pred_test_log = lr_stage1_model.predict(X_test)

# 2. Defang extreme extrapolations (overflow/underflow) and revert to original scale
# Capping at 30.0 ensures expm1 stays within the numerical capacity of float64
SAFE_LOG_MAX = 30.0
X_train_time = np.clip(pred_train_log, a_min=0, a_max=SAFE_LOG_MAX)
X_test_time = np.clip(pred_test_log, a_min=0, a_max=SAFE_LOG_MAX)

# Extract Stage 2 targets into log-space
y_train_time = np.log1p(train_df[target_col_model].to_numpy(dtype=float)).ravel()
y_test_time = np.log1p(test_df[target_col_model].to_numpy(dtype=float)).ravel()

# Initialize Stage 2 Linear Regression Regressor
time_predictor = LinearRegression()

# 3. 5-Fold Cross-Validation Step (on Training Data in log-space)
cv_pred_log = cross_val_predict(time_predictor, X_train_time, y_train_time, cv=5)
cv_mse = mean_squared_error(y_train_time, cv_pred_log)
cv_r2 = r2_score(y_train_time, cv_pred_log)

print("--- 5-Fold Cross-Validation Metrics (Train Set - Log Scale) ---")
print(f"CV MSE = {cv_mse:.4f} | CV R² = {cv_r2:.4f}")

# 4. Final Fit and Test Prediction
time_predictor.fit(X_train_time, y_train_time)
pred_log = time_predictor.predict(X_test_time)

# 5. Inverse-Transform Stage 2 Predictions back to Original Physical Scale
y_test_orig = y_test_time
pred_orig = pred_log

test_mse = mean_squared_error(y_test_orig, pred_orig)
test_r2 = r2_score(y_test_orig, pred_orig)

print("\n--- Final Test Set Metrics (Original Physical Scale) ---")
print(f"MSE = {test_mse:.4f} | R² = {test_r2:.4f}")

# 6. Quantiled Relative Error Distribution (Original Scale)
relative_error = np.abs(pred_orig - y_test_orig) / (np.abs(y_test_orig) + 1e-8)

quantiles = [0.25, 0.50, 0.75]
print("\n--- Quantiled Relative Error Q(q) (Test Set) ---")
print(f"{target_col_model}:")
print(f"Quantile (q) | Relative Error Value Q(q)")

q_values = np.quantile(relative_error, quantiles)
for q, val in zip(quantiles, q_values):
    print(f"  Q({q:.2f}):  {val:.4f}")

In [ ]:
evaluate_model(time_predictor)

### Linear Regression Stage 2 Using the Ground Truth Features From the Empirical Dataset

In [ ]:
X_train_time = train_df[target_cols_reg]
X_test_time = test_df[target_cols_reg]
y_train_time = np.log1p(train_df[target_col_model].to_numpy(dtype=float)).ravel()
y_test_time = np.log1p(test_df[target_col_model].to_numpy(dtype=float)).ravel()

In [ ]:
import os
import joblib
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict

# Initialize Stage 2 Linear Regression Regressor
lr_predictor = LinearRegression()

# 1. 5-Fold Cross-Validation Step (on Training Data in log-space)
cv_pred_log = cross_val_predict(lr_predictor, X_train_time, y_train_time, cv=5)
cv_mse = mean_squared_error(y_train_time, cv_pred_log)
cv_r2 = r2_score(y_train_time, cv_pred_log)

print("--- 5-Fold Cross-Validation Metrics (Train Set - Log Scale) ---")
print(f"CV MSE = {cv_mse:.4f} | CV R² = {cv_r2:.4f}")

# 2. Final Fit and Test Prediction
lr_predictor.fit(X_train_time, y_train_time)
pred_log = lr_predictor.predict(X_test_time)

# 3. Inverse-Transform to Original Physical Scale for Evaluation Metrics
y_test_orig = y_test_time
pred_orig = pred_log

test_mse = mean_squared_error(y_test_orig, pred_orig)
test_r2 = r2_score(y_test_orig, pred_orig)

print("\n--- Final Test Set Metrics (Original Physical Scale) ---")
print(f"MSE = {test_mse:.4f} | R² = {test_r2:.4f}")

# 4. Quantiled Relative Error Distribution (Original Scale)
relative_error = np.abs(pred_orig - y_test_orig) / (np.abs(y_test_orig) + 1e-8)

quantiles = [0.25, 0.50, 0.75]
print("\n--- Quantiled Relative Error Q(q) (Test Set) ---")
print(f"{target_col_model}:")
print(f"Quantile (q) | Relative Error Value Q(q)")

q_values = np.quantile(relative_error, quantiles)
for q, val in zip(quantiles, q_values):
    print(f"  Q({q:.2f}):  {val:.4f}")

# ==============================================================================
# SAVE TRAINED MODEL WEIGHTS
# ==============================================================================
output_dir = str(paths.results_dir() / "retrained_models")
os.makedirs(output_dir, exist_ok=True)

model_path = os.path.join(output_dir, 'best_lr_model_time.pkl')
joblib.dump(lr_predictor, model_path)

print("\n" + "="*50)
print(f"Model successfully saved to: {model_path}")
print("="*50)

In [ ]:
evaluate_model(lr_predictor)

In [ ]:
# ==============================================================================
# COMBINED HYBRID EVALUATION & PERFORMANCE RATIO GENERATION
# ==============================================================================
print("--------------------------------------------------------------------\n" \
    "Performance For Linear Regression Stage 2 Model with predicted features from ground truth features")
# 1. Generate Stage 2 predictions on the test set
test_df["predicted_time"] = lr_predictor.predict(X_test_time)

# 2. Define problem tracking groups
group_cols = ["name", "nvar"]

# 3. Calculate Oracle baseline (Absolute minimum observed time per unique problem)
test_df["oracle_time"] = test_df.groupby(group_cols)["stats_elapsed_time"].transform("min")

# 4. Identify Model Selection (Find configuration with lowest predicted time, map its ACTUAL time)
best_pred_idx = test_df.groupby(group_cols)["predicted_time"].idxmin()
model_time_map = test_df.loc[best_pred_idx].set_index(group_cols)["stats_elapsed_time"]
test_df["best_predicted_time"] = test_df.set_index(group_cols).index.map(model_time_map)

# 5. Compress dataframe down to unique problem instances (the 43 test problems)
test_df_unique = test_df.drop_duplicates(subset=group_cols).copy()

# 6. Calculate sample-by-sample performance ratio (Relative Slowdown)
test_df_unique['model_ratio'] = test_df_unique['best_predicted_time'] / test_df_unique['oracle_time']

# 7. Output target performance metrics for the paper
print("\n" + "="*50)
print("MODEL RATIO QUANTILE SUMMARY (1.0 = PERFECTION)")
print("="*50)
print(test_df_unique['model_ratio'].quantile([0.50, 0.75, 0.90]).to_string())
print("="*50)